# 02 — Build SEDs

Convert AB/Vega magnitudes to F_λ (erg/s/cm²/Å), clip flux outliers, and plot SEDs for each
catalog variant. Consolidates `scripts/seds/DESI_SEDs.py`, `W2M_legacy_SEDs.py`,
`W2M_legacy_multi_SEDs.py`, `At_Once_SEDs.py` into one parameterized conversion function instead
of copy-pasted per-variant scripts.

**Inputs:** `data/matched/*_matched.csv` (from `01_crossmatch.ipynb`)
**Script references:** `scripts/seds/*.py`

The final section (candidate SEDs with template overlay) depends on the UV-excess candidate CSVs
generated in `01_crossmatch.ipynb`; re-run that notebook's candidate-selection cell first if the
crossmatch outputs have changed.

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import yaml
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy.units as u
from synphot import units as su

BASE_DIR = Path.cwd().parent
with open(BASE_DIR / "config" / "qso_params.yaml") as f:
    PARAMS = yaml.safe_load(f)

DATA_MATCHED = BASE_DIR / "data" / "matched"

WAVELENGTHS = PARAMS["band_wavelengths_AA"]
VEGA_TO_AB = PARAMS["vega_to_ab_offsets"]
FLAM_MAX_GALEX_PS1 = PARAMS["flux_outlier_thresholds_flam"]["galex_panstarrs"] * su.FLAM
FLAM_MAX_UKIDSS = PARAMS["flux_outlier_thresholds_flam"]["ukidss"] * su.FLAM


def mag_arr(col):
    if hasattr(col, 'filled'):
        return np.array(col.filled(np.nan), dtype=float)
    arr = np.array(col, dtype=str)
    arr[arr == '--'] = 'nan'
    return arr.astype(float)


def to_flam(table, mag_col, band, vega_offset_key=None, max_flam=FLAM_MAX_GALEX_PS1):
    """Convert an AB (or Vega, via config offset) magnitude column to FLAM,
    clipping values above max_flam to NaN (per config outlier thresholds)."""
    mags = mag_arr(table[mag_col])
    if vega_offset_key is not None:
        mags = mags + VEGA_TO_AB[vega_offset_key]
    lam = WAVELENGTHS[band] * u.AA
    flam = (mags * u.ABmag).to(su.FLAM, u.spectral_density(lam))
    flam[flam > max_flam] = np.nan
    return flam

## DESI-schema SEDs (DESI, W2M-legacy-multi, candidates)

Source: `scripts/seds/DESI_SEDs.py`, `W2M_legacy_multi_SEDs.py`, `At_Once_SEDs.py`. These three
scripts share the same column schema (`FUVmag`, `NUVmag`, `gmag`...`ymag`, UKIDSS `*AperMag3`,
WISE `W*mag`) and only differ in input file and plot style, so they collapse into one function.

In [ ]:
DESI_SCHEMA_BANDS = [
    ('FUVmag', 'FUV', None, FLAM_MAX_GALEX_PS1),
    ('NUVmag', 'NUV', None, FLAM_MAX_GALEX_PS1),
    ('gmag', 'g', None, FLAM_MAX_GALEX_PS1),
    ('rmag', 'r', None, FLAM_MAX_GALEX_PS1),
    ('imag', 'i', None, FLAM_MAX_GALEX_PS1),
    ('zmag', 'z', None, FLAM_MAX_GALEX_PS1),
    ('ymag', 'y', None, FLAM_MAX_GALEX_PS1),
    ('yAperMag3', 'Y', 'Y', FLAM_MAX_UKIDSS),
    ('j_1AperMag3', 'J', 'J', FLAM_MAX_UKIDSS),
    ('hAperMag3', 'H', 'H', FLAM_MAX_UKIDSS),
    ('kAperMag3', 'K', 'K', FLAM_MAX_UKIDSS),
    ('W1mag', 'W1', 'W1', None),
    ('W2mag', 'W2', 'W2', None),
    ('W3mag', 'W3', 'W3', None),
    ('W4mag', 'W4', 'W4', None),
]


def desi_schema_flam(table):
    """Return (lam [AA], flam_by_band dict) for a table using the DESI SED column schema."""
    lam = np.array([WAVELENGTHS[band] for _, band, _, _ in DESI_SCHEMA_BANDS]) * u.AA
    flam = {}
    for mag_col, band, vega_key, max_flam in DESI_SCHEMA_BANDS:
        kwargs = dict(vega_offset_key=vega_key)
        if max_flam is not None:
            kwargs['max_flam'] = max_flam
        flam[band] = to_flam(table, mag_col, band, **kwargs)
    return lam, flam


def is_uv_excess(flam, index):
    f_fn = flam['FUV'][index] - flam['NUV'][index]
    f_ng = flam['NUV'][index] - flam['g'][index]
    f_gr = flam['g'][index] - flam['r'][index]
    upturn = ((f_ng > 0) and (f_gr < 0)) or ((f_fn > 0) and (f_ng < 0))
    return upturn and (flam['FUV'][index] > flam['NUV'][index])


def plot_desi_schema_seds(csv_path, id_col='TARGETID', z_col='Z', only_uv_excess=False, overlay=False):
    table = Table.read(csv_path, format='csv')
    lam, flam = desi_schema_flam(table)
    ids = table[id_col]
    redshift = table[z_col]

    if overlay:
        plt.figure(figsize=(10, 6))
    for index, name in enumerate(ids):
        if only_uv_excess and not is_uv_excess(flam, index):
            continue
        zsp = redshift[index]
        values = [flam[band].value[index] for _, band, _, _ in DESI_SCHEMA_BANDS]
        if not overlay:
            plt.figure()
        plt.plot(lam.value / (1 + zsp), values, marker='o', linestyle='-', alpha=0.7,
                 label=f"{name} (z={zsp:.3f})")
        if not overlay:
            plt.xscale('log')
            plt.yscale('log')
            plt.legend()
            plt.show()
    if overlay:
        plt.xscale('log')
        plt.yscale('log')
        plt.show()

In [ ]:
# DESI: per-source plots, filtered to UV-excess sources only (matches DESI_SEDs.py behavior)
plot_desi_schema_seds(DATA_MATCHED / "UKPSAWG_matched.csv", only_uv_excess=True)

In [ ]:
# W2M-legacy-multi: all sources, one plot per source (matches W2M_legacy_multi_SEDs.py)
w2m_multi_csv = BASE_DIR / "data" / "archive" / "W2M_multi_2arc.csv"
plot_desi_schema_seds(w2m_multi_csv, id_col='designation', z_col='zsp')

## W2M-legacy-raw SEDs

Source: `scripts/seds/W2M_legacy_SEDs.py`. Different column schema (`umag`, `gmag`, `rmag`,
`imag`, `zmag`, `j_m_2mass`/`h_m_2mass`/`k_m_2mass` from 2MASS, `w1mpro`...`w4mpro` from AllWISE)
— separate from the DESI schema above, so it gets its own conversion path.

**Known issue (resolved):** the original script pointed at `data/archive/W2M_QSOs.csv`, which
does not exist. The correct input is `data/raw/W2M_QSOs.csv` (pre-crossmatch) — this notebook
uses the actual crossmatched output, `data/matched/W2M_legacy_COMBINED_matched.csv`, instead,
which additionally has PS1/GALEX/UKIDSS columns from `01_crossmatch.ipynb`.

In [ ]:
# 2MASS Vega->AB offsets are not in config/qso_params.yaml (only UKIDSS is) — using the
# literal values from W2M_legacy_SEDs.py. Flagged for addition to config in a future pass.
TWOMASS_TO_AB = {'J': 0.894, 'H': 1.374, 'K': 1.840}

W2M_LEGACY_RAW_BANDS = [
    ('umag', 3542.0, None),
    ('gmag_1', 4686.0, None),
    ('rmag_1', 6165.0, None),
    ('imag_1', 7481.0, None),
    ('zmag_1', 8923.0, None),
    ('j_m_2mass', 12355.0, TWOMASS_TO_AB['J']),
    ('h_m_2mass', 16458.0, TWOMASS_TO_AB['H']),
    ('k_m_2mass', 21603.0, TWOMASS_TO_AB['K']),
    ('w1mpro', WAVELENGTHS['W1'], VEGA_TO_AB['W1']),
    ('w2mpro', WAVELENGTHS['W2'], VEGA_TO_AB['W2']),
    ('w3mpro', WAVELENGTHS['W3'], VEGA_TO_AB['W3']),
    ('w4mpro', WAVELENGTHS['W4'], VEGA_TO_AB['W4']),
]

w2m_legacy_raw = Table.read(DATA_MATCHED / "W2M_legacy_COMBINED_matched.csv", format='csv')
lam_legacy = np.array([lam for _, lam, _ in W2M_LEGACY_RAW_BANDS]) * u.AA
flam_legacy = []
for mag_col, lam_val, offset in W2M_LEGACY_RAW_BANDS:
    mags = mag_arr(w2m_legacy_raw[mag_col])
    if offset is not None:
        mags = mags + offset
    flam_legacy.append((mags * u.ABmag).to(su.FLAM, u.spectral_density(lam_val * u.AA)))

names = [d[1:5] + d[10:15] for d in np.array(w2m_legacy_raw['designation'], dtype=str)]
redshift_legacy = w2m_legacy_raw['zsp']

for index, name in enumerate(names):
    zsp = redshift_legacy[index]
    plt.figure()
    plt.plot(lam_legacy.value / (1 + zsp),
              [f.value[index] for f in flam_legacy],
              marker='o', label=f"{name} {round(zsp, 5)}")
    plt.xscale('log')
    plt.yscale('log')
    plt.legend()
    plt.show()

## Candidate SEDs with QSO template overlay

**Depends on `01_crossmatch.ipynb` having been run** (reads `uv_excess_candidates*.csv`).

Source: `scripts/seds/COMBINED_SEDs_unred_candidates.py`, `Combined_SEDs_w2m_gri.py`. Plots each
UV-excess candidate's observed photometry alongside the intrinsic QSO template (redshifted,
median-scaled to the observed points), for both DESI and W2M-current candidate rows in the
combined GRI candidate table. Figures are saved to `figures/` (the original script wrote to
`~/Downloads/`, which is not portable).

In [ ]:
import synphot
from astropy.io import ascii
from synphot import SourceSpectrum
from synphot.models import Empirical1D
from synphot.observation import Observation

FILTER_DIR = BASE_DIR / "data" / "filters"
TEMPLATE_FILE = BASE_DIR / "templates" / "qso_template.txt"
FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

_spec = ascii.read(TEMPLATE_FILE)
templateWave = _spec['col1']
templateFlux = _spec['col2']

TWOMASS_TO_AB_2 = {'J': 0.894, 'H': 1.374, 'K': 1.839}  # note: K offset differs slightly (1.839 vs 1.840) from the legacy-raw SED script above; both taken verbatim from their respective source scripts

CAND_BANDS = [
    ('FUVmag', 1549.0, None, FLAM_MAX_GALEX_PS1, "GALEX_GALEX.FUV.dat"),
    ('NUVmag', 2303.0, None, FLAM_MAX_GALEX_PS1, "GALEX_GALEX.NUV.dat"),
    ('gmag', 4810.0, None, FLAM_MAX_GALEX_PS1, "PAN-STARRS_PS1.g.dat"),
    ('rmag', 6170.0, None, FLAM_MAX_GALEX_PS1, "PAN-STARRS_PS1.r.dat"),
    ('imag', 7520.0, None, FLAM_MAX_GALEX_PS1, "PAN-STARRS_PS1.i.dat"),
    ('zmag', 8660.0, None, FLAM_MAX_GALEX_PS1, "PAN-STARRS_PS1.z.dat"),
    ('ymag', 9620.0, None, FLAM_MAX_GALEX_PS1, "PAN-STARRS_PS1.y.dat"),
    ('yAperMag3', 10305.0, 0.634, FLAM_MAX_UKIDSS, "UKIRT_UKIDSS.Y.dat"),
    ('Jmag', 12350.0, TWOMASS_TO_AB_2['J'], FLAM_MAX_UKIDSS, "2MASS_2MASS.J.dat"),
    ('j_1AperMag3', 12483.0, 0.938, FLAM_MAX_UKIDSS, "UKIRT_UKIDSS.J.dat"),
    ('hAperMag3', 16313.0, 1.379, FLAM_MAX_UKIDSS, "UKIRT_UKIDSS.H.dat"),
    ('Hmag', 16620.0, TWOMASS_TO_AB_2['H'], FLAM_MAX_UKIDSS, "2MASS_2MASS.H.dat"),
    ('Kmag', 21590.0, TWOMASS_TO_AB_2['K'], FLAM_MAX_UKIDSS, "2MASS_2MASS.Ks.dat"),
    ('kAperMag3', 22010.0, 1.900, FLAM_MAX_UKIDSS, "UKIRT_UKIDSS.K.dat"),
    ('W1mag', 33526.0, VEGA_TO_AB['W1'], FLAM_MAX_UKIDSS, "WISE_WISE.W1.dat"),
    ('W2mag', 46028.0, VEGA_TO_AB['W2'], FLAM_MAX_UKIDSS, "WISE_WISE.W2.dat"),
    ('W3mag', 115608.0, VEGA_TO_AB['W3'], FLAM_MAX_UKIDSS, "WISE_WISE.W3.dat"),
    ('W4mag', 220883.0, VEGA_TO_AB['W4'], FLAM_MAX_UKIDSS, "WISE_WISE.W4.dat"),
]


def plot_candidate_sed(table, index, name, ra, dec, zsp, save=True):
    lam_all = np.array([lam for _, lam, _, _, _ in CAND_BANDS])
    flam_all = []
    for mag_col, lam_val, offset, max_flam, _ in CAND_BANDS:
        mags = mag_arr(table[mag_col])
        if offset is not None:
            mags = mags + offset
        flam = (mags * u.ABmag).to(su.FLAM, u.spectral_density(lam_val * u.AA))
        flam[flam > max_flam] = np.nan
        flam_all.append(flam.value[index])
    flam_all = np.array(flam_all)

    plt.figure()
    plt.plot(lam_all / (1 + zsp), flam_all, marker='o', linestyle='-', label=str(name))

    sp = SourceSpectrum(Empirical1D, points=templateWave * u.AA,
                         lookup_table=templateFlux * su.FLAM, z=zsp)
    synth_flx = []
    for _, _, _, _, filt_file in CAND_BANDS:
        bp = synphot.SpectralElement.from_file(str(FILTER_DIR / filt_file))
        try:
            obs = Observation(sp, bp, force='extrap')
            synth_flx.append(obs.effstim('flam').value)
        except (synphot.exceptions.DisjointError, synphot.exceptions.SynphotError):
            synth_flx.append(np.nan)
    synth_flx = np.array(synth_flx)

    valid = np.isfinite(flam_all) & np.isfinite(synth_flx) & (synth_flx > 0)
    scale = np.nanmedian(flam_all[valid] / synth_flx[valid]) if valid.any() else 1.0

    plt.plot(templateWave, scale * templateFlux, color='gray', alpha=0.6, label='QSO template')
    lam_tpl_rest = lam_all / (1 + zsp)
    valid_tpl = np.isfinite(synth_flx)
    plt.scatter(lam_tpl_rest[valid_tpl], scale * synth_flx[valid_tpl],
                color='orange', marker='s', zorder=5, label='template synth phot')

    plt.loglog()
    plt.title(f'RA = {ra:.4f}   DEC = {dec:.4f}')
    plt.ylim(1e-18, 2e-16)
    plt.legend(fontsize=8)
    plt.tight_layout()

    if save:
        safe_name = "".join(c if (c.isalnum() or c in "._-") else "_" for c in str(name))
        plt.savefig(FIGURES_DIR / f"SED_{safe_name}.png", dpi=150)
    plt.show()


cand_table = Table.read(DATA_MATCHED / "uv_excess_candidates_w2m_gri.csv", format='csv')

# Loop 1: DESI candidates (Z > 0; Z == 0 indicates a W2M row with no DESI redshift)
desi_z = mag_arr(cand_table['Z'])
for index in range(len(cand_table)):
    if desi_z[index] == 0.0:
        continue
    plot_candidate_sed(cand_table, index, cand_table['TARGETID'][index],
                        mag_arr(cand_table['RA'])[index], mag_arr(cand_table['DEC'])[index], desi_z[index])

# Loop 2: W2M candidates (zsp > 0; zsp == 0 indicates a DESI row)
w2m_zsp = mag_arr(cand_table['zsp'])
for index in range(len(cand_table)):
    if w2m_zsp[index] == 0.0:
        continue
    plot_candidate_sed(cand_table, index, cand_table['designation'][index],
                        mag_arr(cand_table['ra'])[index], mag_arr(cand_table['dec'])[index], w2m_zsp[index])